In [1]:
# Bibliotecas

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix, accuracy_score, f1_score
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn import svm
from sklearn.svm import SVC

In [2]:
df = pd.read_excel('dataset_ultrasound_durrani.xlsx')
df = df.drop(columns='Fiber_Content')
df

,Path_Length,Travel_time_54,UPV_54,Travel_time_250,UPV_250,Configuration,Target
0,0.533,109.3,4880.15,111.3,4788.859,0,1
1,0.445,91.1,4879.25,93.1,4779.807,0,1
2,0.342,68.6,4980.03,72.3,4730.290,0,1
3,0.229,44.2,5171.95,47.2,4851.695,0,1
4,0.499,102.7,4857.41,105.2,4743.346,0,1
...,...,...,...,...,...,...,...
560,0.076,12.2,6245.90,17.3,4393.060,2,28
561,0.152,30.7,4964.17,33.2,4578.310,2,28
562,0.229,46.9,4874.20,52.1,4395.390,2,28
563,0.305,62.2,4900.32,68.4,4459.060,2,28


In [3]:
categorical = pd.get_dummies(df['Configuration'], prefix='Configuration')
df = df.drop(columns='Configuration')
df = pd.concat((df, categorical), axis=1)
df

,Path_Length,Travel_time_54,UPV_54,Travel_time_250,UPV_250,Target,Configuration_0,Configuration_1,Configuration_2
0,0.533,109.3,4880.15,111.3,4788.859,1,True,False,False
1,0.445,91.1,4879.25,93.1,4779.807,1,True,False,False
2,0.342,68.6,4980.03,72.3,4730.290,1,True,False,False
3,0.229,44.2,5171.95,47.2,4851.695,1,True,False,False
4,0.499,102.7,4857.41,105.2,4743.346,1,True,False,False
...,...,...,...,...,...,...,...,...,...
560,0.076,12.2,6245.90,17.3,4393.060,28,False,False,True
561,0.152,30.7,4964.17,33.2,4578.310,28,False,False,True
562,0.229,46.9,4874.20,52.1,4395.390,28,False,False,True
563,0.305,62.2,4900.32,68.4,4459.060,28,False,False,True


In [10]:
df_train, df_test = train_test_split(df, test_size=0.3, random_state=42)

In [22]:
df_train

,Path_Length,Travel_time_54,UPV_54,Travel_time_250,UPV_250,Target,Configuration_0,Configuration_1,Configuration_2
141,0.261,52.8,4940.50,54.3,4806.630,28,True,False,False
299,0.254,52.8,4810.61,51.8,4903.470,3,False,True,False
424,0.254,56.4,4503.55,56.4,4503.550,1,False,True,False
557,0.229,43.7,5231.12,49.6,4616.940,14,False,False,True
19,0.401,81.3,4936.29,82.3,4872.418,3,True,False,False
...,...,...,...,...,...,...,...,...,...
71,0.130,24.3,5330.86,27.8,4676.259,28,True,False,False
106,0.504,98.9,5095.41,103.2,4883.721,7,True,False,False
270,0.036,8.3,4314.94,9.4,3829.787,14,True,False,False
435,0.508,110.8,4584.84,111.3,4564.240,3,False,True,False


In [ ]:
target = 'Target'
attributes = list(df.columns[(df.columns != target)])

scaler = StandardScaler()

scaler.fit(df_train[attributes])

df_train_norm = scaler.transform(df_train[attributes])
df_test_norm = scaler.transform(df_test[attributes])

In [35]:
svm_ovo = SVC(decision_function_shape='ovo', kernel='rbf')

# conjunto de treino
X_train = df_train_norm
y_train = df_train[[target]]
svm_ovo.fit(X_train, y_train)

# conjunto de teste
X_test = df_test_norm
y_test = df_test[[target]]

y_pred = svm_ovo.predict(X_test)

acc = svm_ovo.score(X_test, y_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.81      0.47      0.59        45
           3       0.17      0.32      0.22        28
           7       0.17      0.37      0.23        30
          14       0.29      0.24      0.26        29
          28       0.50      0.03      0.05        38

    accuracy                           0.29       170
   macro avg       0.39      0.28      0.27       170
weighted avg       0.43      0.29      0.29       170



C:\Users\juliafmoura\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [36]:
# Classificador nulo que apenas chuta uma resposta considerando a distribuições das classes
dummy_clf = DummyClassifier(strategy='stratified')
dummy_clf.fit(X_train,y_train)
print(classification_report(y_test, dummy_clf.predict(X_test)))

              precision    recall  f1-score   support

           1       0.35      0.18      0.24        45
           3       0.14      0.18      0.15        28
           7       0.04      0.03      0.04        30
          14       0.11      0.17      0.13        29
          28       0.22      0.21      0.21        38

    accuracy                           0.16       170
   macro avg       0.17      0.15      0.15       170
weighted avg       0.19      0.16      0.16       170



In [37]:
# Validação cruzada estratificada
X = df.drop(columns=['Target']).values
y = df['Target'].values
scaler = StandardScaler()
skf = StratifiedKFold(n_splits=10, random_state=42, shuffle=True)

fold_accuracies = []
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    clf = SVC(decision_function_shape='ovo', kernel='rbf')
    clf.fit(X_train_sc, y_train)

    predictions   = clf.predict(X_test_sc)
    accuracy      = accuracy_score(y_test, predictions)
    fold_accuracies.append(accuracy)

    print(f"Fold {fold:02d}" f"| Acurácia: {accuracy:.4f}")
print(f"Acurácia média: {np.mean(fold_accuracies):.4f} \n")

Fold 01| Acurácia: 0.2281
Fold 02| Acurácia: 0.2281
Fold 03| Acurácia: 0.2807
Fold 04| Acurácia: 0.2456
Fold 05| Acurácia: 0.3158
Fold 06| Acurácia: 0.2857
Fold 07| Acurácia: 0.2679
Fold 08| Acurácia: 0.2500
Fold 09| Acurácia: 0.3929
Fold 10| Acurácia: 0.3571
Acurácia média: 0.2852 

